In [ ]:
import requests
import json
import datetime
import time
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
import pandas as pd
import lxml
import lxml.etree as et
import lxml.html as lh
from bs4 import BeautifulSoup
import html5lib
import csv
import feedparser
import regex as re # https://pypi.org/project/regex/

# progress bar
from IPython.core.interactiveshell import InteractiveShell
from ipywidgets import IntProgress
from IPython.display import display

InteractiveShell.ast_node_interactivity = "none"

def p_bar(mx, desc):
    global p
    p = IntProgress(min=0, max=mx, description=desc)
    return p


In [ ]:
# define path
driver_path = 'chromedriver'  # update this to your local chromedriver path, or leave as-is if chromedriver is on your PATH
# define HTML parser
htmlparser = et.HTMLParser()
# load selenium driver
driver = webdriver.Chrome(driver_path)

In [ ]:
f = open('../output/ADSScraping/greylitlinks.txt')
urls = [url.strip() for url in f.readlines()]

In [ ]:
# set up variables
filename = '../output/ADSScraping/greylitinfopages.json'
data = []
is_last = False
failed = []
id_no = 0

# show progress bar
display(p_bar(len(urls), 'Scraping pages'))

# loop
for ind, url in enumerate(urls):
    entry = {'URL': url}
    try:
        # get page
        driver.get(url)
        # read page with BeautifulSoup
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        # find data table
        table = soup.find('table', {'summary': 'Series data'})
        if table:
            rows = table.tbody.find_all('tr', recursive=False)
            for row in rows:
                label = row.find('th').div.span.div.text.strip()
                value_container = row.find('td')
                
                if label in ['Downloads', 'DOI']:
                    value = value_container.find('a')['href']
                
                elif value_container.find('table'):
                    tbls = value_container.find_all('table')
                    
                    if len(tbls) == 2 or label == 'Identifiers':
                        rws = tbls[0].find_all('tr')
                        value = {}
                        for rw in rws:
                            tds = rw.find_all('td')
                            lbl = tds[0].text.strip()
                            val = tds[1].text
                            val = re.sub(r'[ ]{2,}', ' ', val, 0, 0) # clean multiple spaces
                            value[lbl] = val.strip()
                        
                        if len(tbls) == 2:  
                            gridloc = tbls[1].find('td')
                            
                            if gridloc:
                                gridloc = gridloc.text.strip() # eg: Grid Reference: 293540, 90830 (Easting, Northing)
                                
                                try:
                                    _str = gridloc.split(':')[1].strip()  # ['Grid Reference', '293540, 90830 (Easting, Northing)']
                                    sp1 = _str.split(',')  # ['293540', '90830 (Easting', 'Northing)']
                                    sp2 = sp1[1].strip().split(' ')  # ['90830', '(Easting'] 
                                    value['Grid Reference'] = {
                                        sp2[1].strip()[1:]: sp1[0].strip(),  # Easting: 293540
                                        sp1[2].strip()[:-1]: sp2[0].strip()  # Northing: 90830
                                    }
                                
                                except  Exception as e:
                                    value['Grid Reference'] = gridloc.split(':')[1].strip()
                    
                    else:
                        cells = tbls[0].find_all('td')
                        value = []
                        
                        for cell in cells:
                            val = cell.text.replace('\n', '')
                            val = re.sub(r'[ ]{2,}', ' ', val, 0, 0) # clean multiple spaces
                            value.append(val.strip())
                            
                elif value_container.find('a'):
                    link = value_container.find('a')
                    
                    if link.find('div'):
                        value = {
                            'text': link.find_all('div')[1].text.strip().replace('\n', ''),
                            'url': link['href']
                        }
                    
                    else:
                        value = {
                            'text': link.text.strip(),
                            'url': link['href']
                        }
                
                else:
                    value = value_container.text
                    value = re.sub(r'[ ]{2,}', ' ', value, 0, 0) # clean multiple spaces
                    value = value.strip() # clean leading and trailing spaces

                entry[label] = value
            
            data.append(entry)

        else:
            entry.update({'index': ind, 'error': 'Page could not be loaded.'})
            failed.append(entry)
    
    except Exception as e:
        entry.update({'index': ind, 'error': e})
        failed.append(entry)
    
    # every 10 pages update progress bar
    if ind%10 == 0: 
        p.value += 10  
    
    # every 1000 pages save progress to disk
    if ind%1000 == 0:
        with open(filename, 'w') as file:
            file.write(json.dumps(data))

# write results to file
with open(filename, 'w') as file:
    file.write(json.dumps(data))

# print report
print(f'{len(data)} out of {ind} pages were scraped successfully. {len(failed)} pages failed as follows:\n')
for page in failed:
    print(f'INDEX: {page["index"]}\nURL: {page["URL"]}\nERROR: {page["error"]}\n')

# close driver
driver.quit()


In [ ]:
# open json file
with open(filename, 'r') as file:
    data = file.read()